# Full249 score refit using five valid folds


In [ ]:
from pathlib import Path
import os, json, copy, shutil, warnings
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv

PACKAGE=Path.cwd().parent
PROJECT = Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
BASE_RUN="final_rerun_2026"
NEW_RUN="reviewer_full249_score_5fold"
BASE=PROJECT/"runs"/BASE_RUN
DEST=PROJECT/"runs"/NEW_RUN

os.environ.update(CHD_OUTPUT_ROOT=str(PROJECT),CHD_RUN_ID=BASE_RUN,
    CHD_COHORT_ID="prs_raw106_complete",CHD_SENSITIVITY_OUTPUT_DIR=str(BASE),
    CHD_SHARED_NOTEBOOK=str((Path.cwd() / '00_shared_config.ipynb')),CHD_BOOTSTRAP_N="1000")
get_ipython().run_line_magic("run", '"'+str((Path.cwd() / '00_shared_config.ipynb'))+'"')
tr,va,te,feature_sets,base_manifest=validate_pipeline_artifacts(True)
base_frames={"train":tr,"val":va,"test":te}
oldmeta=base_manifest["nmr_score"]
assert oldmeta["predictor_contract_name"]=="Full249"
assert eid_digest(tr)==oldmeta["train_eid_sha256"]
assert int(oldmeta["inner_folds"])==5
grid_path=BASE/"09_audit/nmr_score_inner_cv_grid.csv"
grid=pd.read_csv(grid_path)
required={"l1_ratio","alpha","mean_inner_uno_c_10y","valid_folds","selected"}
assert required.issubset(grid.columns)
for c in ["l1_ratio","alpha","mean_inner_uno_c_10y","valid_folds"]:
    grid[c]=pd.to_numeric(grid[c],errors="raise")
assert len(grid)==len(oldmeta["l1_ratio_grid"])*len(oldmeta["alpha_grid"])
assert not grid.duplicated(["l1_ratio","alpha"]).any()
assert np.allclose(sorted(grid.l1_ratio.unique()),sorted(oldmeta["l1_ratio_grid"]))
assert np.allclose(sorted(grid.alpha.unique()),sorted(oldmeta["alpha_grid"]),rtol=1e-12,atol=0)
previous=grid.loc[grid.selected.astype(str).str.lower().eq("true")]
assert len(previous)==1
assert np.isclose(previous.iloc[0].alpha,oldmeta["best_alpha"],rtol=1e-12,atol=0)
assert np.isclose(previous.iloc[0].l1_ratio,oldmeta["best_l1_ratio"],rtol=1e-12,atol=0)
assert np.isclose(previous.iloc[0].mean_inner_uno_c_10y,oldmeta["best_score"],rtol=1e-12)
eligible=grid.loc[grid.valid_folds.eq(5)&np.isfinite(grid.mean_inner_uno_c_10y)]
assert len(eligible)>0
# idxmax preserves the original grid order for exact ties.
chosen=grid.loc[eligible.mean_inner_uno_c_10y.idxmax()]
best_l1=float(chosen.l1_ratio); best_alpha=float(chosen.alpha)
assert best_l1>min(oldmeta["l1_ratio_grid"])
assert min(oldmeta["alpha_grid"])<best_alpha<max(oldmeta["alpha_grid"])
print("Old selected candidate:")
display(previous)
print("New selection: maximum mean score among candidates with 5 valid folds:")
display(chosen.to_frame().T)
print("Training EID digest:",eid_digest(tr))
print("New output:",DEST)


In [ ]:
# Refit only the selected model; no repetition of the 5-fold parameter search.
if DEST.exists():
    raise FileExistsError(f"{DEST} already exists. Use the downstream cell if refitting previously succeeded.")
cols=list(full249_cols)
assert len(cols)==249
assert canonical_json_digest(cols)==oldmeta["predictor_field_ids_sha256"]
raw={k:f[cols].replace([np.inf,-np.inf],np.nan).astype(float) for k,f in base_frames.items()}
assert not raw["train"].isna().all().any()
imputer=SimpleImputer(strategy="median")
scaler=StandardScaler()
Xtr=scaler.fit_transform(imputer.fit_transform(raw["train"]))
Xs={"train":Xtr}
for k in ["val","test"]:
    Xs[k]=scaler.transform(imputer.transform(raw[k]))
y=Surv.from_arrays(event=base_frames["train"].Event.astype(bool).to_numpy(),
                   time=base_frames["train"].Time.to_numpy(float))
model=CoxnetSurvivalAnalysis(l1_ratio=best_l1,alphas=[best_alpha],
                            max_iter=NMR_ELASTICNET_MAX_ITER,tol=1e-8)
with warnings.catch_warnings():
    warnings.simplefilter("error",ConvergenceWarning)
    model.fit(Xtr,y)
beta=np.asarray(model.coef_[:,0],float)
assert np.isfinite(beta).all() and not np.allclose(beta,0)
scores={k:X@beta for k,X in Xs.items()}
assert all(np.isfinite(v).all() for v in scores.values())

# Create a separate run only after model fitting succeeds.
DEST.mkdir(parents=True)
shutil.copytree(BASE/"01_cohort",DEST/"01_cohort")
os.environ.update(CHD_RUN_ID=NEW_RUN,CHD_SENSITIVITY_OUTPUT_DIR=str(DEST))
get_ipython().run_line_magic("run", '"'+str((Path.cwd() / '00_shared_config.ipynb'))+'"')

newmeta=copy.deepcopy(oldmeta)
newmeta.update(score_name="NMR_score249_5fold",best_l1_ratio=best_l1,
    best_alpha=best_alpha,best_score=float(chosen.mean_inner_uno_c_10y),
    selected_n=int((np.abs(beta)>1e-12).sum()),selected_valid_folds=5,
    selection_rule="maximum mean inner-CV 10y Uno C-index among valid_folds == 5",
    reference_run=BASE_RUN,downstream_column_alias="NMR_score",
    source_cv_grid=str(grid_path),source_cv_grid_sha256=sha256_file(grid_path),
    previous_selection={"l1_ratio":oldmeta["best_l1_ratio"],"alpha":oldmeta["best_alpha"],
                        "valid_folds":int(previous.iloc[0].valid_folds)})
newmeta["boundary_audit"].update(best_l1_ratio=best_l1,best_alpha=best_alpha,
    l1_at_first_boundary=False,l1_at_last_boundary=bool(best_l1==max(oldmeta["l1_ratio_grid"])),
    alpha_at_first_boundary=False,alpha_at_last_boundary=False,
    any_boundary_hit=bool(best_l1==max(oldmeta["l1_ratio_grid"])),
    search_extension_required=False,
    accepted_parameter_endpoint=bool(best_l1==1.0))
grid["selected_previous"]=grid["selected"]
grid["selected"]=False
grid.loc[chosen.name,"selected"]=True
atomic_write_csv(grid,Path(AUDIT_DIR)/"nmr_score_inner_cv_grid.csv",index=False)
atomic_write_csv(pd.DataFrame([newmeta["boundary_audit"]]),
                 Path(AUDIT_DIR)/"nmr_score_boundary_audit.csv",index=False)
coef=pd.DataFrame({"Contract_Order":np.arange(1,250),"Field_ID":cols,
                  "Coefficient":beta,"Selected":np.abs(beta)>1e-12})
atomic_write_csv(coef,Path(SCORE_ARTIFACT_DIR)/"nmr_score_full249_coefficients.csv",index=False)
prep=pd.DataFrame({"Field_ID":cols,"Train_Median_Imputation":imputer.statistics_,
    "Train_Standardization_Mean":scaler.mean_,"Train_Standardization_Scale":scaler.scale_})
atomic_write_csv(prep,Path(PREPROCESSING_ARTIFACT_DIR)/"nmr_score_full249_train_preprocessing.csv",index=False)
atomic_write_json(newmeta,Path(SCORE_ARTIFACT_DIR)/"nmr_score_metadata.json")
registry_path=Path(SCORE_ARTIFACT_DIR)/"score249_5fold_active_registry.csv"
atomic_write_csv(pd.DataFrame([{"Score":"NMR_score249_5fold","Candidate_N":249,
    "Selected_N":newmeta["selected_n"],"Downstream_Alias":"NMR_score","Fit_Split":"train"}]),
    registry_path,index=False)
manifest=copy.deepcopy(base_manifest)
manifest.update(run_id=NEW_RUN,nmr_score=newmeta,reference_run=BASE_RUN,
                revision_role="Full249 parameter selection requiring five valid folds")
manifest["score_registry"]={"csv":str(registry_path),"csv_sha256":sha256_file(registry_path)}
for k in ["train","val","test"]:
    frame=base_frames[k].copy()
    frame["NMR_score"]=scores[k]
    path=split_artifact_path(k)
    atomic_write_csv(frame,path,index=False)
    saved=pd.read_csv(path,low_memory=False)
    assert eid_digest(saved)==eid_digest(base_frames[k])
    assert np.array_equal(saved.Event.to_numpy(),base_frames[k].Event.to_numpy())
    assert np.allclose(saved.Time.to_numpy(float),base_frames[k].Time.to_numpy(float),
                       rtol=0,atol=1e-12)
    manifest["splits"][k].update(file_sha256=sha256_file(path),
        eid_sha256=eid_digest(saved),outcome_sha256=dataframe_digest(saved,["Eid","Time","Event"]))
atomic_write_json(manifest,PIPELINE_MANIFEST_PATH)
atomic_write_json({"active_score":newmeta,"historical_auxiliary_registries_from":str(BASE),
                   "authoritative_manifest":str(PIPELINE_MANIFEST_PATH)},DEST/"SCORE249_5FOLD_PROVENANCE.json")
df_train,df_val,df_test,final_feature_sets,pipeline_manifest=validate_pipeline_artifacts(True)
print("PASS: Full249 refit and saved splits validated.")
print("Selected nonzero coefficients:",newmeta["selected_n"])
print("Output:",DEST)


In [ ]:
# Run after the preceding cell prints PASS; can also resume in a fresh kernel.
from pathlib import Path
import os
PACKAGE=Path.cwd().parent
PROJECT = Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
NEW_RUN="reviewer_full249_score_5fold"
os.environ.update(CHD_OUTPUT_ROOT=str(PROJECT),CHD_RUN_ID=NEW_RUN,
    CHD_COHORT_ID="prs_raw106_complete",CHD_SENSITIVITY_OUTPUT_DIR=str(PROJECT/"runs"/NEW_RUN),
    CHD_SHARED_NOTEBOOK=str((Path.cwd() / '00_shared_config.ipynb')),CHD_BOOTSTRAP_N="1000")
# If interrupted, retain only stages that have not completed successfully.
for stage in ['02_incremental_value.ipynb', '03_nmr_representation_comparison.ipynb', '04_survival_model_comparison.ipynb', '05_calibration_dca_and_joint_risk.ipynb']:
    print("START",stage,flush=True)
    get_ipython().run_line_magic("run",'"'+str(Path.cwd() / stage)+'"')
    print("DONE",stage,flush=True)
